# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the output file automatically.
from datetime import datetime
datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = "./results/patrolling_comms_" + datetime_str + ".zarr"

output_file = "./results/patrolling_comms_draft2_test12.zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
from multiprocessing import Process
from args import *

MAX_PROCESSES = 25

# Set default arguments for evaluation. Feel free to change these!
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(1)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(3)
default_args["episode_length"] = wrap_arg_val(200)
default_args["seed"] = wrap_arg_val(2)

default_args["observation_mask"] = wrap_arg_val(True)

default_args["diffusion_autoregression_steps"] = wrap_arg_val(1)

# default_args["graph_name"] = wrap_arg_val("cumberland")
# default_args["graph_file"] = wrap_arg_val("../../patrolling_zoo/graphs/cumberland.graph")
default_args["graph_random"] = wrap_arg_val(True)

default_args["observation_radius_random_min"] = wrap_arg_val(0.0)
default_args["observation_radius_random_max"] = wrap_arg_val(0.0)

# Evaluation

In [3]:
# obs_radii = [0.0, 50.0, 100.0, 150.0, 200.0, 250.0, 300.0]
obs_radii = [50.0, 100.0, 150.0, 200.0]
comms_probability = 0.5

import numpy as np

# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")

else:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)

    for obs_radius in obs_radii:
        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-draft1/wandb/run-20260510_230404-ta52omdy/files/episode_54", default_args)
        args["eval_series"] = "MAGEC"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-pred-draft1/wandb/run-20260510_230404-t69m43g4/files/episode_54", default_args)
        args["eval_series"] = "Prediction"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-pred-uqstd-draft1/wandb/run-20260510_230404-lrwsfonw/files/episode_54", default_args)
        args["eval_series"] = "Prediction + UQ"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-draft1/wandb/run-20260510_230404-zzco3nwv/files/episode_54", default_args)
        args["eval_series"] = "Learned Comms. + Prediction"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260510_230404-zm3h3kr7/files/episode_54", default_args)
        args["eval_series"] = f"Learned Comms. + Prediction + UQ"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        start_process(args)

    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Converted arguments: ['--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors', '--action_neighbors_max_degree', '10', '--action_velocity_acceptance_radius', '5.0', '--agent_speed', '40.0', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'mappo', '--alpha', '1.0', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0.0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--communication_model', 'bernoulli', '--communication_probability', '0.5', '--critic_hidden_size', '64', '--critic_lr', '0.0005', '--cuda', '--cuda_deterministic', '--cuda_idx', '1', '--cuda_idx_predictor', '1', '--data_chunk_length', '10', '--no-dec_actor', '--diffusion_autoregression_steps', '1', '--diffusion_epoch', '5', '--diffusion_model_type', 'gnn', '--diffusion_sample_steps', '5', '--disable_observation_global', '--no-enco

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--episode_max', '101', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")
/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--episode_max', '101', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be Falseu are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False





/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--episode_max', '101', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")
/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--episode_max', '101', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.



...

Creating ensemble of 3 predictors.
Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.Creating ensemble of 3 predictors.

Creating ensemble of 3 predictors.
R_Critic: Use CNN: False, Use MLP: TrueCreating ensemble of 3 predictors.

QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.

QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.QmasPolicy: Using diffusion model type gnn with pre

/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/anthony/miniconda3/envs/qmas/lib/pytho

Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.


/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.

QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.

QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.


# Analysis

In [4]:
import zarr
output = zarr.open(output_file, mode='r')
print(output.tree())
print(f"Output file: {os.path.abspath(output_file)}")
print(f"To load results, set in first cell: output_file = '{os.path.abspath(output_file)}'")

/
 ├── Learned Comms. + Prediction
 │   ├── obs_radius_100.0
 │   │   ├── agent_count (3, 200) float32
 │   │   ├── avg_idleness (3, 200) float32
 │   │   ├── communication
 │   │   │   └── requests_made (3, 200) float32
 │   │   ├── deltaSteps (3, 200, 4, 1) float32
 │   │   ├── node_visits
 │   │   │   ├── node_0 (3, 200) float32
 │   │   │   ├── node_1 (3, 200) float32
 │   │   │   ├── node_10 (3, 200) float32
 │   │   │   ├── node_11 (3, 200) float32
 │   │   │   ├── node_12 (3, 200) float32
 │   │   │   ├── node_13 (3, 200) float32
 │   │   │   ├── node_14 (3, 200) float32
 │   │   │   ├── node_15 (3, 200) float32
 │   │   │   ├── node_16 (3, 200) float32
 │   │   │   ├── node_17 (3, 200) float32
 │   │   │   ├── node_18 (3, 200) float32
 │   │   │   ├── node_19 (3, 200) float32
 │   │   │   ├── node_2 (3, 200) float32
 │   │   │   ├── node_20 (3, 200) float32
 │   │   │   ├── node_21 (3, 200) float32
 │   │   │   ├── node_22 (3, 200) float32
 │   │   │   ├── node_23 (3, 200) floa

In [5]:
%reload_ext autoreload
%autoreload 2
from plots import *
import numpy as np

x_key = 'observation_radius'

# Data for plotting
plots = {
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['prediction_error_mean'], axis=tuple(range(1, output[series][exp]['prediction_error_mean'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_trajectory_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'avg_idleness': {
        'title': 'Time-Averaged Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['avg_idleness']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['avg_idleness'], axis=tuple(range(1, output[series][exp]['avg_idleness'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'avg_idleness_final': {
        'title': 'Final Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['avg_idleness'][:, -1]) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['avg_idleness'][:, -1], axis=tuple(range(1, output[series][exp]['avg_idleness'][:, -1].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'avg_idleness_over_time': {
        'title': 'Avg. Idleness vs. Time',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['avg_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        },
        # 'y_std': {
        #     series: {
        #         exp: np.std(output[series][exp]['avg_idleness'], axis=0)
        #         for exp in sorted_experiments(series, output, x_key)
        #     }
        #     for series in output
        # }
    },
    'worst_idleness': {
        'title': 'Time-Averaged Worst Idleness',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['worst_idleness']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['worst_idleness'], axis=tuple(range(1, output[series][exp]['worst_idleness'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'worst_idleness_final': {
        'title': 'Final Worst Idleness',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['worst_idleness'][:, -1]) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'worst_idleness_over_time': {
        'title': 'Worst Idleness vs. Time',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['worst_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        },
        'y_std': {
            series: {
                exp: np.std(output[series][exp]['worst_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        }
    },
    'stddev_idleness': {
        'title': 'Time-Averaged Std. Dev. Idleness',
        'ylabel': 'Std. Dev. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['stddev_idleness']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'stddev_idleness_over_time': {
        'title': 'Std. Dev. Idleness vs. Time',
        'ylabel': 'Std. Dev. Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['stddev_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        },
        'y_std': {
            series: {
                exp: np.std(output[series][exp]['stddev_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        }
    },
    'communication_requests': {
        'title': 'Communication Requests vs. Observation Radius',
        'ylabel': 'Communication Requests',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['communication']['requests_made']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'communication_requests_time': {
        'title': 'Cumulative Communication Requests vs. Time',
        'ylabel': 'Communication Requests',
        'xlabel': 'Time Steps',
        'y': {
            series: {
                exp: np.cumsum(np.mean(output[series][exp]['communication']['requests_made'], axis=0))
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        }
    },
    'prediction_uncertainty_mean': {
        'title': 'Prediction Uncertainty',
        'ylabel': 'Prediction Uncertainty (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_mean': {
        'title': 'Prediction Uncertainty Gap',
        'ylabel': 'Prediction Uncertainty Gap (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in output]

plot(plots, timesteps=timesteps)
